In [1]:
import torch
from datasets import load_dataset, DatasetDict, Audio

common_voice = DatasetDict()
common_voice["train"] = load_dataset("mozilla-foundation/common_voice_17_0", "am", split="train+validation", trust_remote_code=True)
common_voice["test"] = load_dataset("mozilla-foundation/common_voice_17_0", "am", split="test", trust_remote_code=True)

Using the latest cached version of the module from C:\Users\chapp\.cache\huggingface\modules\datasets_modules\datasets\mozilla-foundation--common_voice_17_0\9d10386a731ff6e6ed4ec973a4dc204a9820e8c842fbe388bdba0dd205ed5016 (last modified on Mon Oct 13 11:51:01 2025) since it couldn't be found locally at mozilla-foundation/common_voice_17_0, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from C:\Users\chapp\.cache\huggingface\modules\datasets_modules\datasets\mozilla-foundation--common_voice_17_0\9d10386a731ff6e6ed4ec973a4dc204a9820e8c842fbe388bdba0dd205ed5016 (last modified on Mon Oct 13 11:51:01 2025) since it couldn't be found locally at mozilla-foundation/common_voice_17_0, or remotely on the Hugging Face Hub.


In [2]:
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

In [4]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    "../checkpoints/whisper-69"
)
processor = WhisperProcessor.from_pretrained(
    "../checkpoints/whisper-69"
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [12]:
from IPython.display import Audio as IPAudio  # Add this import

# Always display samples 65-70 for consistent comparison
for i in range(65, 71):
    sample = common_voice["test"][i]
    print(f"🔹 Sample #{i}")
    print("Reference:", sample["sentence"])

    input_features = processor(
        sample["audio"]["array"],
        sampling_rate=sample["audio"]["sampling_rate"],
        return_tensors="pt"
    ).input_features.to("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            forced_decoder_ids=processor.get_decoder_prompt_ids(language="amharic", task="transcribe")
        )

    predicted_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    print("Prediction:", predicted_text)
    print("Duration (sec):", round(len(sample["audio"]["array"]) / sample["audio"]["sampling_rate"], 2))
    display(IPAudio(sample["audio"]["array"], rate=sample["audio"]["sampling_rate"]))  # Changed to IPAudio
    print("-" * 80)


🔹 Sample #65
Reference: ኑሮው ቀን በቀን ይፋጃል።
Prediction: ኖራው ቀን በቃኔፋው ይፋጨን።
Duration (sec): 3.6


--------------------------------------------------------------------------------
🔹 Sample #66
Reference: አንዳንዶች ፊልምና ዶክመንታሪ ይሰራሉ።
Prediction: አንዳንዶች ፊልም እና ደውክመን ተሪ ይሰራሉ።
Duration (sec): 4.5


--------------------------------------------------------------------------------
🔹 Sample #67
Reference: ቀበሮዎቹ ከሊጉ መሰናበት ካልፈለጉ ይህን ጨዋታ ማሸነፍ ግዴታቸው ነው።
Prediction: ጠበረዎቹ ከሊጉ መሰናበት ካልፈለጉ ይህንጃዋት አማሸነፍ ግዴታጨው ነው።
Duration (sec): 6.59


--------------------------------------------------------------------------------
🔹 Sample #68
Reference: ባልየውም ለራሱ እንግዲህ ሚስቴ ‹እህቴ› ብላ ወንድ ይዛብኝ መጣች፡፡
Prediction: ባልየውም ለራሶ እንግድ ይኒስቴ እህት ሄተይ በላው እንዲዛበይ ነጣች፡፡
Duration (sec): 5.98


--------------------------------------------------------------------------------
🔹 Sample #69
Reference: የእንስሳው ጤንነት የተመረመረ እና የሥጋው ምንጭ በደንብ የታወቀ መሆን እንደሚገባው ይመክራሉ።
Prediction: የአንስስሃው ጤንነት የተመረመረና የሥገውም እንጭ በደምቤታውቀ መሆን እንደሚገባዊ መክራሉ።
Duration (sec): 7.2


--------------------------------------------------------------------------------
🔹 Sample #70
Reference: ኢትዮጵያ በአገሪቱ ውስጥ ለጀመረቻቸውና የምጣኔ ሀብት እድገት ያመጣሉ ላለቻቸው ፕሮጀክቶች ወሳኙ ነው
Prediction: ኢትዮጵያ በአገሪቱ እህት ተለጀመረችአቸው ማ የምፍጣኔ ሃብት ዕድገት ያመጣሉ ላደችቸው ፖወች ወሳኙ ነው።
Duration (sec): 8.03


--------------------------------------------------------------------------------


In [5]:
import torch
from tqdm import tqdm
import evaluate

wer_metric = evaluate.load("wer")

predictions, references = [], []
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

for example in tqdm(common_voice["test"]):
    input_features = processor(
        example["audio"]["array"],
        sampling_rate=example["audio"]["sampling_rate"],
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features)

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    predictions.append(transcription)
    references.append(example["sentence"])

wer = 100 * wer_metric.compute(predictions=predictions, references=references)
print(f"Test WER (pretrained model): {wer:.2f}%")

100%|██████████| 205/205 [04:26<00:00,  1.30s/it]

Baseline WER (pretrained model): 69.64%
